# Evaluation (precision, recall, f1-score - Micro, Macro, Weighted)

## Get entities LLM

In [ ]:
#%%
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from get_entities_LLM import extract_llm_entities, entity_vocab
from tqdm import tqdm

#%%
# Load your labeled dataset
df = pd.read_csv("ARP_PreLabel.csv")
df["entities"] = df["Entities"].apply(eval)

#%%
# Split into train and test
train_df, test_df = train_test_split(df, test_size=0.3, random_state=123)
# get sentences and true entities to list
## sentences
test_sentences = test_df["Sentence"].tolist()
## entities
true_entities_list_with_sentiment = test_df["entities"].tolist()
true_entities_list = [[ent[0] for ent in entity_group if ent[0] != ''] for entity_group in true_entities_list_with_sentiment]


### no boost

In [ ]:
#%%
# Wrap original extract_llm_entities with a progress bar (no need to modify .py)
def extract_llm_entities_with_progress(sentences):
    results = []
    for sent in tqdm(sentences, desc="LLM Prediction"):
        result = extract_llm_entities([sent])[0]
        results.append(result)
    return results

In [2]:
#%%
# Run prediction with progress bar
pred_results = extract_llm_entities_with_progress(test_sentences)
pred_entities_list = [res["entities"] for res in pred_results]

LLM Prediction: 100%|██████████| 445/445 [28:07<00:00,  3.79s/it]


### boost

In [2]:
from tqdm import tqdm

BATCH_SIZE = 200  # 可視 rate limit 調 100~500

def extract_llm_entities_in_batches(sentences, batch_size=BATCH_SIZE):
    results = []
    n = len(sentences)
    for start in tqdm(range(0, n, batch_size), desc="LLM Prediction"):
        batch = sentences[start:start+batch_size]
        # 一次丟一大批，讓 extract_llm_entities 內部開並行
        results.extend(extract_llm_entities(batch))
    return results

# 使用
pred_results = extract_llm_entities_in_batches(test_sentences)
pred_entities_list = [res["entities"] for res in pred_results]

LLM Prediction: 100%|██████████| 3/3 [02:21<00:00, 47.03s/it]


### evaluate

In [3]:
#%%
# Convert to multi-hot
def entities_to_multihot(entities, vocab):
    multihot = [0] * len(vocab)
    for ent in entities:
        if ent in vocab:
            multihot[vocab.index(ent)] = 1
    return multihot

y_true = [entities_to_multihot(ents, entity_vocab) for ents in true_entities_list]
y_pred = [entities_to_multihot(ents, entity_vocab) for ents in pred_entities_list]


In [4]:
#%%
# Evaluate
print("Micro F1:", f1_score(y_true, y_pred, average="micro"))
print("Macro F1:", f1_score(y_true, y_pred, average="macro"))
print("Weighted F1:", f1_score(y_true, y_pred, average="weighted"))

print("\nClassification Report:")
print(classification_report(
    y_true, y_pred,
    target_names=entity_vocab,
    zero_division=0
))

Micro F1: 0.6103983794733289
Macro F1: 0.5016426737227713
Weighted F1: 0.6114972944510506

Classification Report:
                           precision    recall  f1-score   support

          Federal Reserve       0.66      0.64      0.65        77
           Interest Rates       0.75      0.55      0.64        49
                Inflation       0.95      0.92      0.93        85
               Employment       0.74      0.79      0.76        33
             Unemployment       0.89      1.00      0.94         8
                      GDP       0.48      0.46      0.47        26
                    Trade       0.75      1.00      0.86         6
                 Congress       0.00      0.00      0.00         0
          Monetary Policy       0.59      0.70      0.64        70
      Financial Stability       0.00      0.00      0.00         0
          Price Stability       0.64      0.41      0.50        17
Regulatory Implementation       0.00      0.00      0.00         2
              